## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [25]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [26]:
MODEL = "gpt-6-luna"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [27]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [28]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [29]:
retriever.invoke("Who is Avery?")

[Document(id='569d4d08-6e9b-4323-aaf3-b4c6d1190b85', metadata={'source': '/Users/admin/projects/udemy/llm_engineering/week5/knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000'),
 Document(id='d714223d-6a76-4618-bea4-5a69b9a67fc6', metadata={'source': '/Users/admin/projects/udemy/llm_engineering/week5/knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="- **2010 - 2013**: Business Analyst at Edge Analytics  \n  Prior to joining Innovate, Avery worked as a Business Analyst, focusing on market trends and consumer preferences in the insurance space. This position laid the groundwork for Avery’s future entrepreneurial endeavors.\n\n## Annual Performance History\n- **2015**: **Exceeds Expectations**  \n  Avery’s leadership 

In [30]:
llm.invoke("Who is Avery?")

AIMessage(content='Which Avery do you mean? There are many people with that name—if you share a bit of context, I can help identify them.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 10, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 73, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EdDaFAZJeh2OBnPvDSPkUF7VPUERf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--7f6288e5-5c4a-4ec1-ab7d-e0d3a997de49-0', usage_metadata={'input_tokens': 10, 'output_tokens': 110, 'total_tokens': 120, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 73}})

## Time to put this together!

In [31]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [32]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [33]:
answer_question("Who is Averi Lancaster?", [])

'You may mean **Avery Lancaster**. She co-founded Insurellm in 2015 and serves as its CEO.'

## What could possibly come next? 😂

In [34]:
gr.ChatInterface(answer_question).launch()

/Users/admin/projects/udemy/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!